In [ ]:
import json
import os

import numpy as np
import pandas as pd
from planqk.api.client import PlanqkApiClient

In [ ]:
np.random.seed(42)
X_train = pd.DataFrame({"f0": np.random.randn(20), "f1": np.random.randn(20), "f2": np.random.randn(20)})
y_train = pd.DataFrame({"target": [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]})
X_test = pd.DataFrame({"f0": np.random.randn(20), "f1": np.random.randn(20), "f2": np.random.randn(20)})
y_test = pd.DataFrame({"target": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]})

dataset = {
    "training_tabular_data": X_train.to_dict(),
    "training_target_data": y_train.to_dict(),
    "test_tabular_data": X_test.to_dict(),
    "test_target_data": y_test.to_dict(),
}

with open("data.json", "w") as f:
    json.dump(dataset, f)

api = PlanqkApiClient(
    access_token=os.environ["PLANQK_ACCESS_TOKEN"],
    organization_id=os.environ["PLANQK_ORGANIZATION_ID"],
)

input_dp = api.api.data_pools.create_data_pool(name="Rimay Bug Report Input")
with open("data.json", "rb") as f:
    api.api.data_pools.add_data_pool_file(id=input_dp.id, file=("data.json", f))

output_dp = api.api.data_pools.create_data_pool(name="Rimay Bug Report Output")
print(f"{output_dp=}")

In [ ]:
os.environ["PLANQK_CONSUMER_KEY"] = "5LLB51S2RFZWZHLR477Y"
os.environ["PLANQK_CONSUMER_SECRET"] = "F8yfsQasEGt5ZXsQzRAu4GZ5CBcZI8ChvZWHyIBB"

In [ ]:
from planqk.service.client import PlanqkServiceClient

client = PlanqkServiceClient(
    service_endpoint=os.environ["PLANQK_SERVICE_ENDPOINT"],
    access_key_id=os.environ["PLANQK_CONSUMER_KEY"],
    secret_access_key=os.environ["PLANQK_CONSUMER_SECRET"],
)

service_input = {
    "input_data_pool": {
        "id": "762050f6-84f0-4d6d-a61b-25fec2795643",
        "ref": "DATAPOOL",
    },
    "output_data_pool": {
        "id": "b1d82018-a0ac-4251-9c4c-9668ddad7a95",
        "ref": "DATAPOOL",
    },
    "num_shots": 100,
    "num_runs": 1,
}

execution = client.run(request=service_input)
print(f"{execution=}")
result = execution.result()  # Blocks until completion
print(f"{result=}")

In [ ]:
result

In [ ]:
datapool_client = PlanqkApiClient(
    organization_id=os.getenv("PLANQK_ORGANIZATION_ID"),
    access_token=os.getenv("PLANQK_ACCESS_TOKEN"),
)

output_datapool_id = "b1d82018-a0ac-4251-9c4c-9668ddad7a95"

# List files in output datapool
files = datapool_client.data_pools.get_data_pool_files(id=output_datapool_id)

download_dir = "quantum_results"
os.makedirs(download_dir, exist_ok=True)

# Download all result files
for file_info in files:
    file_content_stream = datapool_client.data_pools.get_data_pool_file(id=output_datapool_id, file_id=file_info.id)
    file_path = os.path.join(download_dir, file_info.name)
    with open(file_path, "wb") as f:
        for chunk in file_content_stream:
            f.write(chunk)

# Load quantum features (contains single-body expectations + two-body correlations)
Xq_train = np.load(os.path.join(download_dir, "1_Xq_train_0.npy"))
Xq_test = np.load(os.path.join(download_dir, "1_Xq_validation_0.npy"))
yq_train = np.load(os.path.join(download_dir, "1_yq_train_0.npy"))
yq_test = np.load(os.path.join(download_dir, "1_yq_validation_0.npy"))

print(f"Quantum training features shape: {Xq_train.shape}")
print(f"Quantum test features shape: {Xq_test.shape}")
print("Features include single-body expectations and two-body correlations")

In [ ]:
Xq_train